In [13]:
import pandas as pd
import numpy as np

def extract_data():
    """
    Extracts raw datasets from the local disk using absolute file paths.
    """
    print("-> Extracting raw datasets from local path...")
    logs = pd.read_csv(r'C:\Users\berkay\OneDrive - student.cankaya.edu.tr\Masaüstü\damn_belgeler\uk\herts\data eng\project\PerfectPetClub_server_log.csv')
    crm = pd.read_csv(r'C:\Users\berkay\OneDrive - student.cankaya.edu.tr\Masaüstü\damn_belgeler\uk\herts\data eng\project\PerfectPetClub_CRM_report.csv')
    survey = pd.read_csv(r'C:\Users\berkay\OneDrive - student.cankaya.edu.tr\Masaüstü\damn_belgeler\uk\herts\data eng\project\PerfectPetClub_survey.csv')
    return logs, crm, survey


def transform_data(logs: pd.DataFrame, crm: pd.DataFrame, survey: pd.DataFrame) -> pd.DataFrame:
    """
    Performs Data Quality checks, Data Cleaning, Feature Engineering, and Merging.
    """
    print("-> Transforming and cleaning datasets...")
    
    # 1. Clean & Feature Engineer: Server Logs
    logs['datetime'] = pd.to_datetime(logs['datetime'])
    logs.drop_duplicates(inplace=True)
    logs['is_mobile'] = logs['user_agent'].str.contains('Mobile', case=False, na=False).astype(int)
    
    max_log_date = logs['datetime'].max()
    
    log_features = logs.groupby('login').agg(
        total_sessions=('session_id', 'count'),
        days_since_last_login=('datetime', lambda x: (max_log_date - x.max()).days),
        mobile_session_ratio=('is_mobile', 'mean')
    ).reset_index().rename(columns={'login': 'email'})
    
    # 2. Clean & Feature Engineer: CRM Report
    crm['recieved_datatime'] = pd.to_datetime(crm['recieved_datatime'])
    crm.drop_duplicates(inplace=True)
    crm['platform_clean'] = crm['platform'].str.replace(r"[\[\]']", "", regex=True)
    
    crm_features = crm.groupby('customer_id').agg(
        total_complaints=('type', lambda x: (x == 'complaint').sum()),
        total_queries=('type', lambda x: (x == 'query').sum())
    ).reset_index()

    # 3. Clean & Feature Engineer: Survey Data
    survey.drop_duplicates(inplace=True)
    
    def parse_income(val):
        """Converts income range strings (e.g., '26k-30k') into numeric midpoints (e.g., 28000.0)."""
        if pd.isna(val): 
            return np.nan
        clean_val = str(val).replace('k', '000').replace('+', '')
        parts = clean_val.split('-')
        if len(parts) == 2:
            return (float(parts[0]) + float(parts[1])) / 2
        return float(parts[0])

    survey['income_numeric'] = survey['reported_income'].apply(parse_income)
    
    # Define Target Variable: Churn (1 = Low NPS/High Churn Risk, 0 = Retained)
    if 'nps_recommend' in survey.columns:
        survey['Churn'] = (survey['nps_recommend'] <= 6).astype(int)
    else:
        survey['Churn'] = 0

    # 4. Merge Datasets
    final_df = survey.merge(log_features, on='email', how='left')
    
    # 5. Missing Value Imputation & Quality Checks
    final_df['total_sessions'].fillna(0, inplace=True)
    final_df['days_since_last_login'].fillna(final_df['days_since_last_login'].max(), inplace=True)
    final_df['mobile_session_ratio'].fillna(0, inplace=True)
    final_df['income_numeric'].fillna(final_df['income_numeric'].median(), inplace=True)
    
    print(f"-> Transformation complete. Processed dataset shape: {final_df.shape}")
    return final_df


def load_data(df: pd.DataFrame, output_filepath: str = 'final_churn_dataset.parquet'):
    """
    Exports the transformed dataset into a Parquet file.
    """
    print(f"-> Saving processed dataset to Parquet format: {output_filepath}")
    df.to_parquet(output_filepath, index=False, engine='pyarrow')
    print("-> ETL Pipeline completed successfully!")


# --- Pipeline Execution ---
raw_logs, raw_crm, raw_survey = extract_data()
processed_dataset = transform_data(raw_logs, raw_crm, raw_survey)
load_data(processed_dataset, 'final_churn_dataset.parquet')

-> Extracting raw datasets from local path...
-> Transforming and cleaning datasets...
-> Transformation complete. Processed dataset shape: (221, 10)
-> Saving processed dataset to Parquet format: final_churn_dataset.parquet
-> ETL Pipeline completed successfully!


C:\Users\berkay\AppData\Local\Temp\ipykernel_16732\296805185.py:69: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_df['total_sessions'].fillna(0, inplace=True)
C:\Users\berkay\AppData\Local\Temp\ipykernel_16732\296805185.py:70: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For

In [18]:
import pandas as pd

# Parquet dosyasını okuma
df = pd.read_parquet('final_churn_dataset.parquet')

# İlk 5 satırı görüntüleme
df.head()

,email,reported_income,nps_recommend,satisfaction_score,online_comparison_freq,income_numeric,Churn,total_sessions,days_since_last_login,mobile_session_ratio
0,Sargsyan701@somemail.co.uk,26k-30k,8,6,5,28000.0,0,48,303,0.0
1,Mammadova282@example.com,34k-38k,5,8,9,36000.0,1,1085,0,0.0
2,Sana858@notrealemail.com,26k-30k,9,5,5,28000.0,0,624,0,0.0
3,Omar.Farooq4@example.com,198k-200k+,9,3,5,199000.0,0,1540,0,0.0
4,Alejandro357@somemail.co.uk,42k-46k,0,8,10,44000.0,1,388,210,1.0


In [19]:
import pandas as pd

# Pandas'ın tüm sütunları ve satırları göstermesini sağla
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100) # İstersen 500 ya da len(df) yapabilirsin

# Parquet dosyasını okuyup tamamını gör
df = pd.read_parquet('final_churn_dataset.parquet')
df

,email,reported_income,nps_recommend,satisfaction_score,online_comparison_freq,income_numeric,Churn,total_sessions,days_since_last_login,mobile_session_ratio
0,Sargsyan701@somemail.co.uk,26k-30k,8,6,5,28000.0,0,48,303,0.0
1,Mammadova282@example.com,34k-38k,5,8,9,36000.0,1,1085,0,0.0
2,Sana858@notrealemail.com,26k-30k,9,5,5,28000.0,0,624,0,0.0
3,Omar.Farooq4@example.com,198k-200k+,9,3,5,199000.0,0,1540,0,0.0
4,Alejandro357@somemail.co.uk,42k-46k,0,8,10,44000.0,1,388,210,1.0
...,...,...,...,...,...,...,...,...,...,...
216,Mitchell201@example.com,26k-30k,10,3,7,28000.0,0,10,0,0.0
217,Priya-Deshmukh562@somemail.co.uk,46k-50k,10,1,8,48000.0,0,17,0,0.0
218,Devendra-Bhat33@somemail.co.uk,74k-78k,8,7,3,76000.0,0,9,0,0.0
219,J_Park76@fakemail.co.uk,22k-26k,6,8,7,24000.0,1,15,0,0.0


In [20]:
display(df)

,email,reported_income,nps_recommend,satisfaction_score,online_comparison_freq,income_numeric,Churn,total_sessions,days_since_last_login,mobile_session_ratio
0,Sargsyan701@somemail.co.uk,26k-30k,8,6,5,28000.0,0,48,303,0.0
1,Mammadova282@example.com,34k-38k,5,8,9,36000.0,1,1085,0,0.0
2,Sana858@notrealemail.com,26k-30k,9,5,5,28000.0,0,624,0,0.0
3,Omar.Farooq4@example.com,198k-200k+,9,3,5,199000.0,0,1540,0,0.0
4,Alejandro357@somemail.co.uk,42k-46k,0,8,10,44000.0,1,388,210,1.0
...,...,...,...,...,...,...,...,...,...,...
216,Mitchell201@example.com,26k-30k,10,3,7,28000.0,0,10,0,0.0
217,Priya-Deshmukh562@somemail.co.uk,46k-50k,10,1,8,48000.0,0,17,0,0.0
218,Devendra-Bhat33@somemail.co.uk,74k-78k,8,7,3,76000.0,0,9,0,0.0
219,J_Park76@fakemail.co.uk,22k-26k,6,8,7,24000.0,1,15,0,0.0


In [17]:
# Sadece senin gözle kontrol etmen için geçici Excel/CSV çıktısı
df.to_csv('trainable_data.csv', index=False)

In [ ]:
"""testing"""

In [ ]:
""""

import os
import sqlite3
import pandas as pd


def process_perfect_pet_club_data(db_path, output_csv_path):
    # 1. Connect to SQLite Database
    conn = sqlite3.connect(db_path)

    # 2. Extract Data with Explicit SQL Joins and Column Aliasing

    

    query = """


""""
    SELECT 
        s.id AS sale_id,
        s.order_date,
        s.expected_date,
        s.delivered_date,
        
        c.id AS customer_id,
        c.name AS customer_name,
        c.email AS customer_email,
        c.county AS customer_county,
        c.postcode AS customer_postcode,
        
        sp.product_id,
        sp.quantity,
        
        p.name AS product_name,
        p.price AS unit_price,
        
        pc.id AS category_id,
        pc.name AS category_name,
        
        pay.payment_gateway,
        pay.amount AS payment_amount,
        
        sub.start_date AS sub_start_date,
        sub.end_date AS sub_end_date,
        sub.auto_renew AS sub_auto_renew,
        st.name AS subscription_plan,
        st.delivery_service
        
    FROM sale s
    JOIN customer c ON s.customer_id = c.id
    JOIN sale_product sp ON s.id = sp.sale_id
    JOIN product p ON sp.product_id = p.id
    LEFT JOIN product_category pc ON p.product_category_id = pc.id
    LEFT JOIN payment pay ON s.payment_id = pay.id
    LEFT JOIN subscription sub ON c.subscription_id = sub.id
    LEFT JOIN subscription_type st ON sub.subscription_type_id = st.id
    """

"""


    df = pd.read_sql_query(query, conn)
    conn.close()

    # 3. Handle Category Imputation (Fill NULL categories for Subscription products)
    df["category_name"] = df["category_name"].fillna("Subscription Plan")
    df["category_id"] = df["category_id"].fillna(-1).astype(int)

    # 4. Datetime Formatting
    date_cols = [
        "order_date",
        "expected_date",
        "delivered_date",
        "sub_start_date",
        "sub_end_date",
    ]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    # 5. Feature Engineering & Business Logic Edge Cases
    # A. Total Price per Line Item
    df["total_line_item_price"] = df["quantity"] * df["unit_price"]

    # B. Identify Subscription Purchases (No delivery physical movement required)
    is_subscription = df["product_name"].str.contains(
        "subscription", case=False, na=False
    )
    df["is_subscription_item"] = is_subscription.astype(int)

    # C. Delivery Duration (Days)
    df["delivery_duration_days"] = (
        df["delivered_date"] - df["order_date"]
    ).dt.days
    df.loc[is_subscription, "delivery_duration_days"] = -1

    # D. Delivery Delay Relative to Expected Date
    df["delivery_delay_days"] = (
        df["delivered_date"] - df["expected_date"]
    ).dt.days
    df.loc[is_subscription, "delivery_delay_days"] = 0

    # E. Categorical Delivery Status
    df["delivery_status"] = "Delivered On Time"

    # Delayed condition
    df.loc[
        (~is_subscription) & (df["delivery_delay_days"] > 0), "delivery_status"
    ] = "Delayed"

    # Pending / In-Transit condition for physical products with missing delivered dates
    df.loc[
        (~is_subscription) & (df["delivered_date"].isna()), "delivery_status"
    ] = "In Transit / Pending"

    # Subscription plan purchases
    df.loc[is_subscription, "delivery_status"] = "Subscription Service"

    # F. Missing Value Imputation
    df["payment_gateway"] = df["payment_gateway"].fillna("Unknown")

    # 6. Save Processed Master Dataset
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df.to_csv(output_csv_path, index=False)

    print(f"Data Processing Completed successfully!")
    print(
        f"Exported {len(df)} records to processed dataset path: '{output_csv_path}'"
    )
    return df


if __name__ == "__main__":
    # Notebook'un bulunduğu klasörü tam yol olarak al
    BASE_DIR = os.path.dirname(os.path.abspath("deneme.ipynb"))
    DB_PATH = os.path.join(BASE_DIR, "PerfectPetClub.db")

    # Var olan 'data' klasörünün doğrudan içine kaydetmek için:
    OUTPUT_PATH = os.path.join(BASE_DIR, "..", "data", "processed_sales.csv")

    print("Veritabanı Yolu:", DB_PATH)
    print("Çıktı CSV Yolu:", OUTPUT_PATH)

    df = process_perfect_pet_club_data(
        db_path=DB_PATH, output_csv_path=OUTPUT_PATH
    )

    df.query("is_subscription_item == 1")[
    ["product_name", "category_id", "category_name"]
].head()    

"""